In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

try:
    results = pd.read_csv(url)
    print("Downloaded latest data from GitHub")
except Exception:
    results = pd.read_csv("./raw_data/results.csv")
    print("Using local backup")

url = "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv"

try:
    shootouts = pd.read_csv(url)
    print("Downloaded latest data from GitHub")
except Exception:
    shootouts = pd.read_csv("./raw_data/shootouts.csv")
    print("Using local backup")

url = "https://raw.githubusercontent.com/martj42/international_results/master/former_names.csv"

try:
    former_names = pd.read_csv(url)
    print("Downloaded latest data from GitHub")
except Exception:
    former_names = pd.read_csv("./raw_data/former_names.csv")
    print("Using local backup")

# Convert dates to datetime
results["date"] = pd.to_datetime(results["date"])
shootouts["date"] = pd.to_datetime(shootouts["date"])

# Merge shootout information
merged = results.merge(
    shootouts,
    on=["date", "home_team", "away_team"],
    how="left"
)

# Create a flag indicating whether the match went to penalties
merged["shootout"] = merged["winner"].notna().astype(int)

# Save intermediate file
merged.to_csv("./results/results_with_shootouts.csv", index=False)

print(f"Results rows: {len(results):,}")
print(f"Merged rows: {len(merged):,}")
print(f"Shootouts found: {merged['shootout'].sum():,}")

Downloaded latest data from GitHub
Downloaded latest data from GitHub
Downloaded latest data from GitHub
Results rows: 49,477
Merged rows: 49,477
Shootouts found: 677


In [2]:
print(former_names.head())
print(former_names.columns)

          current                former  start_date    end_date
0           Benin               Dahomey  1959-11-08  1975-11-30
1    Burkina Faso           Upper Volta  1960-04-14  1984-08-04
2         Curaçao  Netherlands Antilles  1957-03-03  2010-10-10
3  Czechoslovakia               Bohemia  1903-04-05  1919-01-01
4  Czechoslovakia   Bohemia and Moravia  1939-01-01  1945-05-01
Index(['current', 'former', 'start_date', 'end_date'], dtype='object')


Update former_names versus current names
This is not totally correct, as some countries split (e.g. Czechoslovakia)

In [3]:
name_map = dict(
    zip(former_names["former"], former_names["current"])
)

merged["home_team"] = merged["home_team"].replace(name_map)
merged["away_team"] = merged["away_team"].replace(name_map)

merged.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,winner,first_shooter,shootout
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,NaN,NaN,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,NaN,NaN,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,NaN,NaN,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,NaN,NaN,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,NaN,NaN,0


Calculate Elo rating (different for friendlies), Elo difference between teams and win probability

In [4]:
import numpy as np

# Load data
df = merged.copy()

# Prepare dates
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

def get_k_factor(tournament):
    tournament = str(tournament)

    if tournament == "Friendly":
        return 10

    if tournament == "FIFA World Cup":
        return 60

    if tournament in [
        "UEFA Euro",
        "Copa América",
        "African Cup of Nations",
        "AFC Asian Cup",
        "Gold Cup",
        "Oceania Nations Cup"
    ]:
        return 50

    if "World Cup qualification" in tournament:
        return 40

    if "qualification" in tournament:
        return 30

    if "Nations League" in tournament:
        return 30

    if tournament == "Confederations Cup":
        return 40

    return 20

# Elo settings
START_ELO = 1500
K = 32 # quite high, makes it more responsive to recent results
HOME_ADVANTAGE = 80 # between 65-100

# Store current Elo rating for each team
elo_ratings = {}

def get_elo(team):
    return elo_ratings.get(team, START_ELO)

def expected_score(team_elo, opponent_elo):
    return 1 / (1 + 10 ** ((opponent_elo - team_elo) / 400))

def actual_score(home_score, away_score):
    if home_score > away_score:
        return 1, 0
    elif home_score < away_score:
        return 0, 1
    else:
        return 0.5, 0.5

home_elos = []
away_elos = []
elo_diffs = []
home_win_probs = []

for _, row in df.iterrows():
    home_team = row["home_team"]
    away_team = row["away_team"]

    home_elo = get_elo(home_team)
    away_elo = get_elo(away_team)

    # Save pre-match Elo ratings
    home_elos.append(home_elo)
    away_elos.append(away_elo)
    elo_diffs.append(home_elo - away_elo)

    # Home advantage only if not neutral
    neutral = row.get("neutral", False)
    home_adv = 0 if neutral else HOME_ADVANTAGE

    expected_home = expected_score(home_elo + home_adv, away_elo)
    expected_away = 1 - expected_home

    home_win_probs.append(expected_home)

    actual_home, actual_away = actual_score(
        row["home_score"],
        row["away_score"]
    )

    # Update Elo ratings after match
    match_k = get_k_factor(row["tournament"])

    elo_ratings[home_team] = home_elo + match_k * (actual_home - expected_home)
    elo_ratings[away_team] = away_elo + match_k * (actual_away - expected_away)

# Add Elo features to dataframe
df["home_elo"] = home_elos
df["away_elo"] = away_elos
df["elo_difference"] = elo_diffs
df["home_elo_win_probability"] = home_win_probs

# Add result label
df["result"] = np.where(
    df["home_score"] > df["away_score"], "HomeWin",
    np.where(df["home_score"] < df["away_score"], "AwayWin", "Draw")
)

# Save result
df.to_csv("./results/international_matches_with_elo.csv", index=False)

print("Saved: international_matches_with_elo.csv")
print(df[[
    "date", "home_team", "away_team",
    "home_score", "away_score",
    "home_elo", "away_elo",
    "elo_difference",
    "home_elo_win_probability",
    "result"
]].head())

Saved: international_matches_with_elo.csv
        date home_team away_team  home_score  away_score     home_elo  \
0 1872-11-30  Scotland   England         0.0         0.0  1500.000000   
1 1873-03-08   England  Scotland         4.0         2.0  1501.131368   
2 1874-03-07  Scotland   England         2.0         1.0  1495.030850   
3 1875-03-06   England  Scotland         2.0         2.0  1500.963971   
4 1876-03-04  Scotland   England         3.0         0.0  1500.193689   

      away_elo  elo_difference  home_elo_win_probability   result  
0  1500.000000        0.000000                  0.613137     Draw  
1  1498.868632        2.262736                  0.616222  HomeWin  
2  1504.969150       -9.938299                  0.599482  HomeWin  
3  1499.036029        1.927941                  0.615766     Draw  
4  1499.806311        0.387378                  0.613666  HomeWin  


More feature engineering:
- tournament importance: a friendly game isn't as reliable as a World Cup game, adjust
- recent momentum

In [5]:
tournaments = sorted(df["tournament"].dropna().unique())

importance_map = {
    tournament: 5
    for tournament in tournaments
}

for k, v in importance_map.items():
    print(f'"{k}": {v},')

"ABCS Tournament": 5,
"AFC Asian Cup": 5,
"AFC Asian Cup qualification": 5,
"AFC Challenge Cup": 5,
"AFC Challenge Cup qualification": 5,
"AFC Solidarity Cup": 5,
"AFF Championship": 5,
"AFF Championship qualification": 5,
"ASEAN Championship": 5,
"ASEAN Championship qualification": 5,
"African Cup of Nations": 5,
"African Cup of Nations qualification": 5,
"African Friendship Games": 5,
"Afro-Asian Games": 5,
"Al Ain International Cup": 5,
"All-African Games": 5,
"Amílcar Cabral Cup": 5,
"Arab Cup": 5,
"Arab Cup qualification": 5,
"Asian Games": 5,
"Atlantic Cup": 5,
"Atlantic Heritage Cup": 5,
"Balkan Cup": 5,
"Baltic Cup": 5,
"Beijing International Friendship Tournament": 5,
"Benedikt Fontana Cup": 5,
"Bolivarian Games": 5,
"Brazil Independence Cup": 5,
"British Home Championship": 5,
"CAFA Nations Cup": 5,
"CCCF Championship": 5,
"CECAFA Cup": 5,
"CFU Caribbean Cup": 5,
"CFU Caribbean Cup qualification": 5,
"CONCACAF Championship": 5,
"CONCACAF Championship qualification": 5,
"CONCA

Make importance list

In [6]:
def assign_tournament_importance(tournament):
    tournament = str(tournament)

    # Lowest importance
    if tournament == "Friendly":
        return 1

    # Highest importance
    if tournament == "FIFA World Cup":
        return 10

    # Major continental finals
    major_continental = [
        "UEFA Euro",
        "Copa América",
        "African Cup of Nations",
        "AFC Asian Cup",
        "Gold Cup",
        "Oceania Nations Cup",
        "World Cup qualification"
    ]

    if tournament in major_continental:
        return 8

    # Continental qualification
    if "qualification" in tournament:
        return 4

    # Nations League / Confederations-style competitions
    if "Nations League" in tournament or tournament == "Confederations Cup":
        return 6

    # Olympic Games
    if tournament == "Olympic Games":
        return 5

    # Everything else
    return 3


df["tournament_importance"] = df["tournament"].apply(
    assign_tournament_importance
)

tournament_summary = (
    df.groupby(["tournament", "tournament_importance"])
      .size()
      .reset_index(name="n_matches")
      .sort_values("n_matches", ascending=False)
)

print(tournament_summary.head(20))

                               tournament  tournament_importance  n_matches
94                               Friendly                      1      18388
89           FIFA World Cup qualification                      4       8771
184               UEFA Euro qualification                      4       2824
11   African Cup of Nations qualification                      4       2327
88                         FIFA World Cup                     10       1036
54                           Copa América                      8        869
10                 African Cup of Nations                      8        845
2             AFC Asian Cup qualification                      4        829
185                   UEFA Nations League                      6        658
31                             CECAFA Cup                      3        620
33        CFU Caribbean Cup qualification                      4        606
127                    Merdeka Tournament                      3        599
28          

Build a baseline model: logistic regression

In [7]:
from collections import defaultdict, deque

# Include tournament_importance
df["tournament_importance"] = (
    df["tournament"]
      .map(importance_map)
      .fillna(5)
)

# Settings
FORM_WINDOW = 5 # checks last 5 games
GOALS_WINDOW = 10 # When calculating average goals, use last 10 games

# Store team history before each match
team_points_history = defaultdict(lambda: deque(maxlen=FORM_WINDOW))
team_goals_for_history = defaultdict(lambda: deque(maxlen=GOALS_WINDOW))
team_goals_against_history = defaultdict(lambda: deque(maxlen=GOALS_WINDOW))
team_match_count = defaultdict(int)

# New feature columns
home_form_5 = []
away_form_5 = []

home_avg_goals_for_10 = []
away_avg_goals_for_10 = []

home_avg_goals_against_10 = []
away_avg_goals_against_10 = []

home_matches_played = []
away_matches_played = []


def mean_or_default(values, default=0):
    if len(values) == 0:
        return default
    return np.mean(values)

def get_match_points(goals_for, goals_against):
    if goals_for > goals_against:
        return 3
    elif goals_for == goals_against:
        return 1
    else:
        return 0

# Create pre-match features

for _, row in df.iterrows():
    home = row["home_team"]
    away = row["away_team"]

    home_score = row["home_score"]
    away_score = row["away_score"]

    # ---- Save features BEFORE updating histories ----

    home_form_5.append(sum(team_points_history[home]))
    away_form_5.append(sum(team_points_history[away]))

    home_avg_goals_for_10.append(
        mean_or_default(team_goals_for_history[home])
    )
    away_avg_goals_for_10.append(
        mean_or_default(team_goals_for_history[away])
    )

    home_avg_goals_against_10.append(
        mean_or_default(team_goals_against_history[home])
    )
    away_avg_goals_against_10.append(
        mean_or_default(team_goals_against_history[away])
    )

    home_matches_played.append(team_match_count[home])
    away_matches_played.append(team_match_count[away])

    # ---- Update histories AFTER the match ----

    home_points = get_match_points(home_score, away_score)
    away_points = get_match_points(away_score, home_score)

    team_points_history[home].append(home_points)
    team_points_history[away].append(away_points)

    team_goals_for_history[home].append(home_score)
    team_goals_for_history[away].append(away_score)

    team_goals_against_history[home].append(away_score)
    team_goals_against_history[away].append(home_score)

    team_match_count[home] += 1
    team_match_count[away] += 1

# Add features to dataframe

df["home_form_5"] = home_form_5
df["away_form_5"] = away_form_5
df["form_difference_5"] = df["home_form_5"] - df["away_form_5"]

df["home_avg_goals_for_10"] = home_avg_goals_for_10
df["away_avg_goals_for_10"] = away_avg_goals_for_10
df["goals_for_difference_10"] = (
    df["home_avg_goals_for_10"] - df["away_avg_goals_for_10"]
)

df["home_avg_goals_against_10"] = home_avg_goals_against_10
df["away_avg_goals_against_10"] = away_avg_goals_against_10
df["goals_against_difference_10"] = (
    df["home_avg_goals_against_10"] - df["away_avg_goals_against_10"]
)

df["home_matches_played"] = home_matches_played
df["away_matches_played"] = away_matches_played
df["experience_difference"] = (
    df["home_matches_played"] - df["away_matches_played"]
)

# Make sure neutral is numeric
df["neutral"] = df["neutral"].astype(int)

# Create result target if not already present
df["result"] = np.where(
    df["home_score"] > df["away_score"],
    "HomeWin",
    np.where(
        df["home_score"] < df["away_score"],
        "AwayWin",
        "Draw"
    )
)

# Save feature-engineered dataset

df.to_csv("./results/international_matches_features.csv", index=False)

print("Saved: international_matches_features.csv")
print(df[[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "home_elo",
    "away_elo",
    "elo_difference",
    "home_form_5",
    "away_form_5",
    "form_difference_5",
    "home_avg_goals_for_10",
    "away_avg_goals_for_10",
    "goals_for_difference_10",
    "home_avg_goals_against_10",
    "away_avg_goals_against_10",
    "goals_against_difference_10",
    "neutral",
    "tournament",
    "tournament_importance",
    "result"
]].head(20))

Saved: international_matches_features.csv
         date         home_team away_team  home_score  away_score  \
0  1872-11-30          Scotland   England         0.0         0.0   
1  1873-03-08           England  Scotland         4.0         2.0   
2  1874-03-07          Scotland   England         2.0         1.0   
3  1875-03-06           England  Scotland         2.0         2.0   
4  1876-03-04          Scotland   England         3.0         0.0   
5  1876-03-25          Scotland     Wales         4.0         0.0   
6  1877-03-03           England  Scotland         1.0         3.0   
7  1877-03-05             Wales  Scotland         0.0         2.0   
8  1878-03-02          Scotland   England         7.0         2.0   
9  1878-03-23          Scotland     Wales         9.0         0.0   
10 1879-01-18           England     Wales         2.0         1.0   
11 1879-04-05           England  Scotland         5.0         4.0   
12 1879-04-07             Wales  Scotland         0.0        

Add momentum. ELO means something, but recent changes are probably more important

How much stronger or weaker is a team now compared with 10 matches ago?

In [8]:
LOOKBACK = 10

elo_history = defaultdict(lambda: deque(maxlen=LOOKBACK + 1))

home_elo_trend = []
away_elo_trend = []

for _, row in df.iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]

    home_elo = row["home_elo"]
    away_elo = row["away_elo"]

    # Home trend
    if len(elo_history[home_team]) >= LOOKBACK:
        trend = home_elo - elo_history[home_team][0]
    else:
        trend = 0

    home_elo_trend.append(trend)

    # Away trend
    if len(elo_history[away_team]) >= LOOKBACK:
        trend = away_elo - elo_history[away_team][0]
    else:
        trend = 0

    away_elo_trend.append(trend)

    # Update history
    elo_history[home_team].append(home_elo)
    elo_history[away_team].append(away_elo)

df["home_elo_trend"] = home_elo_trend
df["away_elo_trend"] = away_elo_trend

df["elo_trend_difference"] = (
    df["home_elo_trend"] -
    df["away_elo_trend"]
)

df.to_csv(
    "./results/international_matches_features.csv",
    index=False
)

print(
    df[[
        "home_team",
        "away_team",
        "home_elo",
        "away_elo",
        "home_elo_trend",
        "away_elo_trend",
        "elo_trend_difference"
    ]].head(20)
)

           home_team away_team     home_elo     away_elo  home_elo_trend  \
0           Scotland   England  1500.000000  1500.000000        0.000000   
1            England  Scotland  1501.131368  1498.868632        0.000000   
2           Scotland   England  1495.030850  1504.969150        0.000000   
3            England  Scotland  1500.963971  1499.036029        0.000000   
4           Scotland   England  1500.193689  1499.806311        0.000000   
5           Scotland     Wales  1504.057033  1500.000000        0.000000   
6            England  Scotland  1495.942967  1507.870417        0.000000   
7              Wales  Scotland  1496.186616  1513.837714        0.000000   
8           Scotland   England  1519.725477  1489.975670        0.000000   
9           Scotland     Wales  1523.196569  1490.298852        0.000000   
10           England     Wales  1486.504579  1486.868713        0.000000   
11           England  Scotland  1490.378184  1526.626709        0.000000   
12          

Check for missing data

In [9]:
# Check missing values
print(df.isna().sum())

date                               0
home_team                          0
away_team                          0
home_score                        72
away_score                        72
tournament                         0
city                               0
country                            0
neutral                            0
winner                         48800
first_shooter                  49221
shootout                           0
home_elo                           0
away_elo                           0
elo_difference                     0
home_elo_win_probability           0
result                             0
tournament_importance              0
home_form_5                        0
away_form_5                        0
form_difference_5                  0
home_avg_goals_for_10             48
away_avg_goals_for_10             48
goals_for_difference_10           48
home_avg_goals_against_10         48
away_avg_goals_against_10         48
goals_against_difference_10       48
h

Several of these (home_avg_goals_for_10, away_avg_goals_for_10, goals_for_difference_10,home_avg_goals_against_10) are the games that I want to predict! Not include in model!
Winner, first_shooter often unknown, ignore this feature

In [10]:
goal_features = [
    "home_avg_goals_for_10",
    "away_avg_goals_for_10",
    "goals_for_difference_10",
    "home_avg_goals_against_10",
    "away_avg_goals_against_10",
    "goals_against_difference_10"
]

known_matches = df[df["home_score"].notna() & df["away_score"].notna()].copy()
future_matches = df[df["home_score"].isna() | df["away_score"].isna()].copy()

def team_recent_goal_stats(team, known_matches, n=10):
    team_matches = known_matches[
        (known_matches["home_team"] == team) |
        (known_matches["away_team"] == team)
    ].sort_values("date").tail(n)

    goals_for = []
    goals_against = []

    for _, row in team_matches.iterrows():
        if row["home_team"] == team:
            goals_for.append(row["home_score"])
            goals_against.append(row["away_score"])
        else:
            goals_for.append(row["away_score"])
            goals_against.append(row["home_score"])

    return {
        "avg_for": sum(goals_for) / len(goals_for) if goals_for else 0,
        "avg_against": sum(goals_against) / len(goals_against) if goals_against else 0
    }

for idx, row in future_matches.iterrows():
    home = row["home_team"]
    away = row["away_team"]

    home_stats = team_recent_goal_stats(home, known_matches, n=10)
    away_stats = team_recent_goal_stats(away, known_matches, n=10)

    df.loc[idx, "home_avg_goals_for_10"] = home_stats["avg_for"]
    df.loc[idx, "away_avg_goals_for_10"] = away_stats["avg_for"]

    df.loc[idx, "home_avg_goals_against_10"] = home_stats["avg_against"]
    df.loc[idx, "away_avg_goals_against_10"] = away_stats["avg_against"]

    df.loc[idx, "goals_for_difference_10"] = (
        home_stats["avg_for"] - away_stats["avg_for"]
    )

    df.loc[idx, "goals_against_difference_10"] = (
        home_stats["avg_against"] - away_stats["avg_against"]
    )

# Recreate future_matches after filling
future_matches = df[df["home_score"].isna() | df["away_score"].isna()].copy()

print(future_matches[goal_features].isna().sum())

home_avg_goals_for_10          0
away_avg_goals_for_10          0
goals_for_difference_10        0
home_avg_goals_against_10      0
away_avg_goals_against_10      0
goals_against_difference_10    0
dtype: int64


Baseline model: logistic regression
Train only on known_matches

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Train/Test split by date

train = known_matches[known_matches["date"] < "2020-01-01"]
test = known_matches[known_matches["date"] >= "2020-01-01"]

print("Train matches:", len(train))
print("Test matches:", len(test))

# Features
numeric_features = [
    "home_elo",
    "away_elo",
    "elo_difference",

    "home_elo_trend",
    "away_elo_trend",
    "elo_trend_difference",

    "home_form_5",
    "away_form_5",
    "form_difference_5",

    "home_avg_goals_for_10",
    "away_avg_goals_for_10",
    "goals_for_difference_10",

    "home_avg_goals_against_10",
    "away_avg_goals_against_10",
    "goals_against_difference_10",

    "experience_difference",

    "neutral",

    "tournament_importance"
]

categorical_features = [
    "tournament"
]


# Target

X_train = train[numeric_features + categorical_features]
X_test = test[numeric_features + categorical_features]

y_train = train["result"]
y_test = test["result"]


# Preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# Create model

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
    ])

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__solver": ["lbfgs", "newton-cg"],
    "classifier__class_weight": [None, "balanced"]
}

tscv = TimeSeriesSplit(n_splits=5)

grid = GridSearchCV(
    model,
    param_grid,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

# Train

grid.fit(X_train, y_train)

print("\nBest parameters:")
print(grid.best_params_)

print("\nBest cross-validation accuracy:")
print(grid.best_score_)

# Predict
best_model = grid.best_estimator_
predictions = best_model.predict(X_test)

# Evaluate

print("\nAccuracy:")
print(accuracy_score(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

# =====================================================
# Example probabilities
# =====================================================

probs = best_model.predict_proba(X_test)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions,
    "Prob_AwayWin": probs[:, list(best_model.classes_).index("AwayWin")],
    "Prob_Draw": probs[:, list(best_model.classes_).index("Draw")],
    "Prob_HomeWin": probs[:, list(best_model.classes_).index("HomeWin")]
})

print("\nSample predictions:")
print(results.head(20))

Train matches: 43375
Test matches: 6030
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters:
{'classifier__C': 0.1, 'classifier__class_weight': None, 'classifier__solver': 'newton-cg'}

Best cross-validation accuracy:
0.5735509752386222

Accuracy:
0.6029850746268657

Classification Report:
              precision    recall  f1-score   support

     AwayWin       0.58      0.60      0.59      1763
        Draw       0.41      0.01      0.02      1392
     HomeWin       0.61      0.89      0.73      2875

    accuracy                           0.60      6030
   macro avg       0.54      0.50      0.45      6030
weighted avg       0.56      0.60      0.52      6030


Confusion Matrix:
[[1055   10  698]
 [ 460   17  915]
 [ 297   14 2564]]

Sample predictions:
     Actual Predicted  Prob_AwayWin  Prob_Draw  Prob_HomeWin
0   AwayWin   AwayWin      0.679439   0.212506      0.108054
1   AwayWin   AwayWin      0.920142   0.071093      0.008766
2   AwayWin   AwayWin  

In [12]:
X_future = future_matches[numeric_features + categorical_features]

future_predictions = best_model.predict(X_future)
future_probs = best_model.predict_proba(X_future)

future_output = future_matches[[
    "date", "home_team", "away_team", "tournament", "neutral"
]].copy()

future_output["predicted_result"] = future_predictions

classes = list(best_model.classes_)

future_output["prob_away_win"] = future_probs[:, classes.index("AwayWin")]
future_output["prob_draw"] = future_probs[:, classes.index("Draw")]
future_output["prob_home_win"] = future_probs[:, classes.index("HomeWin")]

future_output.to_csv("./results/world_cup_predictions_LogReg.csv", index=False)

print(future_output.head(20))

            date      home_team               away_team      tournament  \
49405 2026-06-11    South Korea          Czech Republic  FIFA World Cup   
49406 2026-06-11         Mexico            South Africa  FIFA World Cup   
49407 2026-06-12         Canada  Bosnia and Herzegovina  FIFA World Cup   
49408 2026-06-12  United States                Paraguay  FIFA World Cup   
49409 2026-06-13          Qatar             Switzerland  FIFA World Cup   
49410 2026-06-13         Brazil                 Morocco  FIFA World Cup   
49411 2026-06-13          Haiti                Scotland  FIFA World Cup   
49412 2026-06-13      Australia                  Turkey  FIFA World Cup   
49413 2026-06-14        Germany                 Curaçao  FIFA World Cup   
49414 2026-06-14    Ivory Coast                 Ecuador  FIFA World Cup   
49415 2026-06-14    Netherlands                   Japan  FIFA World Cup   
49416 2026-06-14         Sweden                 Tunisia  FIFA World Cup   
49417 2026-06-15        B

Try random forest

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

# Hyperparameter Search
rf_param_grid = {
    "classifier__n_estimators": [300],
    "classifier__max_depth": [5,10,15,None],
    "classifier__min_samples_leaf": [3,10],
    "classifier__min_samples_split": [3],
    "classifier__max_features": ["sqrt","log2"],
    "classifier__class_weight": [None,"balanced"]
}

# Time-aware CV
rf_tscv = TimeSeriesSplit(n_splits=5)

rf_grid = GridSearchCV(
    rf_model,
    rf_param_grid,
    cv=rf_tscv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

# Fit the model

rf_grid.fit(X_train, y_train)

print("\nBest RF Parameters:")
print(rf_grid.best_params_)

print("\nBest RF CV Accuracy:")
print(rf_grid.best_score_)

# Predict
best_rf_model = rf_grid.best_estimator_
rf_predictions = best_rf_model.predict(X_test)

print("\nRF Test Accuracy:")
print(accuracy_score(y_test, rf_predictions))

print("\nRF Classification Report:")
print(classification_report(y_test, rf_predictions))

print("\nRF Confusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))

Fitting 5 folds for each of 32 candidates, totalling 160 fits
[CV] END classifier__class_weight=None, classifier__max_depth=5, classifier__max_features=sqrt, classifier__min_samples_leaf=10, classifier__min_samples_split=3, classifier__n_estimators=300; total time=   2.1s
[CV] END classifier__class_weight=None, classifier__max_depth=5, classifier__max_features=sqrt, classifier__min_samples_leaf=3, classifier__min_samples_split=3, classifier__n_estimators=300; total time=   2.3s
[CV] END classifier__class_weight=None, classifier__max_depth=5, classifier__max_features=sqrt, classifier__min_samples_leaf=3, classifier__min_samples_split=3, classifier__n_estimators=300; total time=   3.5s
[CV] END classifier__class_weight=None, classifier__max_depth=5, classifier__max_features=sqrt, classifier__min_samples_leaf=10, classifier__min_samples_split=3, classifier__n_estimators=300; total time=   3.8s
[CV] END classifier__class_weight=None, classifier__max_depth=5, classifier__max_features=sqrt, 

In [14]:
X_future = future_matches[numeric_features + categorical_features]

future_predictions = best_rf_model.predict(X_future)
future_probs = best_rf_model.predict_proba(X_future)

future_output = future_matches[[
    "date", "home_team", "away_team", "tournament", "neutral"
]].copy()

future_output["predicted_result"] = future_predictions

classes = list(best_rf_model.classes_)

future_output["prob_away_win"] = future_probs[:, classes.index("AwayWin")]
future_output["prob_draw"] = future_probs[:, classes.index("Draw")]
future_output["prob_home_win"] = future_probs[:, classes.index("HomeWin")]

future_output.to_csv(
    "./results/world_cup_predictions_RandomForest.csv",
    index=False
)

print(future_output.head(20))

            date      home_team               away_team      tournament  \
49405 2026-06-11    South Korea          Czech Republic  FIFA World Cup   
49406 2026-06-11         Mexico            South Africa  FIFA World Cup   
49407 2026-06-12         Canada  Bosnia and Herzegovina  FIFA World Cup   
49408 2026-06-12  United States                Paraguay  FIFA World Cup   
49409 2026-06-13          Qatar             Switzerland  FIFA World Cup   
49410 2026-06-13         Brazil                 Morocco  FIFA World Cup   
49411 2026-06-13          Haiti                Scotland  FIFA World Cup   
49412 2026-06-13      Australia                  Turkey  FIFA World Cup   
49413 2026-06-14        Germany                 Curaçao  FIFA World Cup   
49414 2026-06-14    Ivory Coast                 Ecuador  FIFA World Cup   
49415 2026-06-14    Netherlands                   Japan  FIFA World Cup   
49416 2026-06-14         Sweden                 Tunisia  FIFA World Cup   
49417 2026-06-15        B

XGBoost model. In first attempt, I noticed the model never predicted draws. To solve this, I've given Draws extra weight

In [16]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_enc
)

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_param_grid = {
    "classifier__n_estimators": [300, 500],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.03, 0.05],
    "classifier__subsample": [0.8],
    "classifier__colsample_bytree": [0.8],
    "classifier__min_child_weight": [1, 5]
}

xgb_grid = GridSearchCV(
    xgb_model,
    xgb_param_grid,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

xgb_grid.fit(X_train, y_train_enc, classifier__sample_weight=sample_weights)

print("\nBest XGBoost parameters:")
print(xgb_grid.best_params_)

print("\nBest XGBoost CV accuracy:")
print(xgb_grid.best_score_)

best_xgb_model = xgb_grid.best_estimator_

xgb_predictions = best_xgb_model.predict(X_test)

print("\nXGBoost Test Accuracy:")
print(accuracy_score(y_test_enc, xgb_predictions))

print("\nXGBoost Classification Report:")
print(classification_report(
    y_test_enc,
    xgb_predictions,
    target_names=le.classes_
))

print("\nXGBoost Confusion Matrix:")
print(confusion_matrix(y_test_enc, xgb_predictions))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=3, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__subsample=0.8; total time=   1.3s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=3, classifier__min_child_weight=1, classifier__n_estimators=300, classifier__subsample=0.8; total time=   1.5s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=3, classifier__min_child_weight=5, classifier__n_estimators=300, classifier__subsample=0.8; total time=   1.9s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=3, classifier__min_child_weight=1, classifier__n_estimators=500, classifier__subsample=0.8; total time=   2.2s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.03, classifier__max_depth=3, classif

In [17]:
X_future = future_matches[numeric_features + categorical_features]

future_predictions_num = best_xgb_model.predict(X_future).astype(int)

future_predictions = le.inverse_transform(future_predictions_num)

future_probs = best_xgb_model.predict_proba(X_future)

future_output = future_matches[[
    "date", "home_team", "away_team", "tournament", "neutral"
]].copy()

future_output["predicted_result"] = future_predictions

future_output["prob_away_win"] = future_probs[:, list(le.classes_).index("AwayWin")]
future_output["prob_draw"] = future_probs[:, list(le.classes_).index("Draw")]
future_output["prob_home_win"] = future_probs[:, list(le.classes_).index("HomeWin")]

future_output.to_csv(
    "./results/world_cup_predictions_XGBoost_optimised.csv",
    index=False
)

print(future_output.head(20))

            date      home_team               away_team      tournament  \
49405 2026-06-11    South Korea          Czech Republic  FIFA World Cup   
49406 2026-06-11         Mexico            South Africa  FIFA World Cup   
49407 2026-06-12         Canada  Bosnia and Herzegovina  FIFA World Cup   
49408 2026-06-12  United States                Paraguay  FIFA World Cup   
49409 2026-06-13          Qatar             Switzerland  FIFA World Cup   
49410 2026-06-13         Brazil                 Morocco  FIFA World Cup   
49411 2026-06-13          Haiti                Scotland  FIFA World Cup   
49412 2026-06-13      Australia                  Turkey  FIFA World Cup   
49413 2026-06-14        Germany                 Curaçao  FIFA World Cup   
49414 2026-06-14    Ivory Coast                 Ecuador  FIFA World Cup   
49415 2026-06-14    Netherlands                   Japan  FIFA World Cup   
49416 2026-06-14         Sweden                 Tunisia  FIFA World Cup   
49417 2026-06-15        B

CatBoost

In [18]:
numeric_features = [...]
categorical_features = ["tournament"]

from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)

cat_features = [
    X_train.columns.get_loc(col)
    for col in categorical_features
]

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    verbose=100,
    random_state=42
)

model.fit(
    X_train,
    y_train_enc,
    cat_features=cat_features
)

0:	learn: 1.0889344	total: 84.2ms	remaining: 1m 24s
100:	learn: 0.9096476	total: 2.08s	remaining: 18.5s
200:	learn: 0.8984207	total: 3.49s	remaining: 13.9s
300:	learn: 0.8928119	total: 4.88s	remaining: 11.3s
400:	learn: 0.8876740	total: 6.31s	remaining: 9.42s
500:	learn: 0.8822644	total: 7.78s	remaining: 7.75s
600:	learn: 0.8770765	total: 9.3s	remaining: 6.17s
700:	learn: 0.8719708	total: 10.8s	remaining: 4.59s
800:	learn: 0.8671667	total: 12.2s	remaining: 3.04s
900:	learn: 0.8625122	total: 13.7s	remaining: 1.5s
999:	learn: 0.8577809	total: 15.1s	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='MultiClass', iterations=1000, learning_rate=0.03, loss_function='MultiClass', random_state=42, verbose=100)

In [19]:
predictions = model.predict(X_test)

from sklearn.metrics import accuracy_score

print(
    "Accuracy:",
    accuracy_score(y_test_enc, predictions)
)

Accuracy: 0.6029850746268657


In [20]:
future_predictions_num = model.predict(X_future).astype(int).flatten()

future_predictions = le.inverse_transform(
    future_predictions_num
)

future_probs = model.predict_proba(X_future)

import pandas as pd
import numpy as np

future_output = future_matches[
    ["date", "home_team", "away_team", "tournament", "neutral"]
].copy()

future_output["predicted_result"] = future_predictions

# Get column positions of each class
away_idx = np.where(le.classes_ == "AwayWin")[0][0]
draw_idx = np.where(le.classes_ == "Draw")[0][0]
home_idx = np.where(le.classes_ == "HomeWin")[0][0]

future_output["prob_away_win"] = future_probs[:, away_idx]
future_output["prob_draw"] = future_probs[:, draw_idx]
future_output["prob_home_win"] = future_probs[:, home_idx]

# Most confident prediction
future_output["confidence"] = future_probs.max(axis=1)

# Sort by confidence
future_output = future_output.sort_values(
    "confidence",
    ascending=False
)

future_output.to_csv(
    "./results/world_cup_predictions_catboost.csv",
    index=False
)

print(future_output.head(20))

            date    home_team               away_team      tournament  \
49475 2026-06-27       Jordan               Argentina  FIFA World Cup   
49413 2026-06-14      Germany                 Curaçao  FIFA World Cup   
49419 2026-06-15        Spain              Cape Verde  FIFA World Cup   
49451 2026-06-23      England                   Ghana  FIFA World Cup   
49436 2026-06-19       Brazil                   Haiti  FIFA World Cup   
49443 2026-06-21        Spain            Saudi Arabia  FIFA World Cup   
49406 2026-06-11       Mexico            South Africa  FIFA World Cup   
49448 2026-06-22       France                    Iraq  FIFA World Cup   
49473 2026-06-27       Panama                 England  FIFA World Cup   
49407 2026-06-12       Canada  Bosnia and Herzegovina  FIFA World Cup   
49421 2026-06-16    Argentina                 Algeria  FIFA World Cup   
49425 2026-06-17     Portugal                DR Congo  FIFA World Cup   
49449 2026-06-23     Portugal              Uzbekist

Try to predict the scores

In [21]:
import os
from scipy.stats import poisson
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error

# =========================
# Load data
# =========================

df = pd.read_csv("./results/international_matches_features.csv")
df["date"] = pd.to_datetime(df["date"])

known_matches = df[df["home_score"].notna() & df["away_score"].notna()].copy()
future_matches = df[df["home_score"].isna() | df["away_score"].isna()].copy()

# =========================
# Features
# =========================

numeric_features = [
    "home_elo",
    "away_elo",
    "elo_difference",

    "home_form_5",
    "away_form_5",
    "form_difference_5",

    "home_avg_goals_for_10",
    "away_avg_goals_for_10",
    "goals_for_difference_10",

    "home_avg_goals_against_10",
    "away_avg_goals_against_10",
    "goals_against_difference_10",

    "neutral",
    "tournament_importance",

    "home_elo_trend",
    "away_elo_trend",
    "elo_trend_difference"
]

categorical_features = ["tournament"]
features = numeric_features + categorical_features

# =========================
# Train/test split
# =========================

train = known_matches[known_matches["date"] < "2020-01-01"].copy()
test = known_matches[known_matches["date"] >= "2020-01-01"].copy()

X_train = train[features]
X_test = test[features]
X_future = future_matches[features]

y_home_train = train["home_score"]
y_away_train = train["away_score"]

y_home_test = test["home_score"]
y_away_test = test["away_score"]

# =========================
# Preprocessor
# =========================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# =========================
# Base Poisson pipeline
# =========================

poisson_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", PoissonRegressor(max_iter=2000))
])

param_grid = {
    "model__alpha": [0.001, 0.01, 0.05, 0.1, 0.5, 1, 2]
}

tscv = TimeSeriesSplit(n_splits=5)

# =========================
# Optimise home-goal model
# =========================

home_grid = GridSearchCV(
    poisson_pipeline,
    param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

home_grid.fit(X_train, y_home_train)

home_goal_model = home_grid.best_estimator_

print("Best home-goal alpha:")
print(home_grid.best_params_)

# =========================
# Optimise away-goal model
# =========================

away_grid = GridSearchCV(
    poisson_pipeline,
    param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

away_grid.fit(X_train, y_away_train)

away_goal_model = away_grid.best_estimator_

print("Best away-goal alpha:")
print(away_grid.best_params_)

# =========================
# Test performance
# =========================

home_lambda_test = home_goal_model.predict(X_test)
away_lambda_test = away_goal_model.predict(X_test)

print("Home goals MAE:", mean_absolute_error(y_home_test, home_lambda_test))
print("Away goals MAE:", mean_absolute_error(y_away_test, away_lambda_test))

# =========================
# Helper functions
# =========================

def score_probabilities(home_lambda, away_lambda, max_goals=8):
    rows = []

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            prob = (
                poisson.pmf(home_goals, home_lambda) *
                poisson.pmf(away_goals, away_lambda)
            )

            rows.append({
                "predicted_home_score": home_goals,
                "predicted_away_score": away_goals,
                "scoreline": f"{home_goals}-{away_goals}",
                "score_probability": prob
            })

    scores = pd.DataFrame(rows)

    # Normalise because we truncate at max_goals
    scores["score_probability"] = (
        scores["score_probability"] /
        scores["score_probability"].sum()
    )

    return scores.sort_values("score_probability", ascending=False)

# =========================
# Predict future games
# =========================

future_home_lambda = home_goal_model.predict(X_future)
future_away_lambda = away_goal_model.predict(X_future)

all_predictions = []

for i, row in future_matches.reset_index(drop=True).iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]

    home_lambda = future_home_lambda[i]
    away_lambda = future_away_lambda[i]

    scores = score_probabilities(
        home_lambda,
        away_lambda,
        max_goals=8
    )

    most_likely = scores.iloc[0]

    home_win_prob = scores[
        scores["predicted_home_score"] > scores["predicted_away_score"]
    ]["score_probability"].sum()

    draw_prob = scores[
        scores["predicted_home_score"] == scores["predicted_away_score"]
    ]["score_probability"].sum()

    away_win_prob = scores[
        scores["predicted_home_score"] < scores["predicted_away_score"]
    ]["score_probability"].sum()

    if home_win_prob > draw_prob and home_win_prob > away_win_prob:
        predicted_result = "HomeWin"
        score_to_submit = scores[
            scores["predicted_home_score"] >
            scores["predicted_away_score"]
        ].iloc[0]

    elif away_win_prob > draw_prob and away_win_prob > home_win_prob:
        predicted_result = "AwayWin"
        score_to_submit = scores[
            scores["predicted_home_score"] <
            scores["predicted_away_score"]
        ].iloc[0]

    else:
        predicted_result = "Draw"
        score_to_submit = scores[
            scores["predicted_home_score"] ==
            scores["predicted_away_score"]
        ].iloc[0]

    predicted_winner = (
        home_team if predicted_result == "HomeWin"
        else away_team if predicted_result == "AwayWin"
        else "Draw"
    )

    all_predictions.append({
        "date": row["date"],
        "home_team": home_team,
        "away_team": away_team,
        "tournament": row["tournament"],
        "neutral": row["neutral"],

        "expected_home_goals": round(home_lambda, 2),
        "expected_away_goals": round(away_lambda, 2),
        "expected_score": f"{round(home_lambda, 1)}-{round(away_lambda, 1)}",

        "most_likely_score": most_likely["scoreline"],
        "most_likely_score_probability": round(most_likely["score_probability"], 4),

        "predicted_result": predicted_result,
        "predicted_winner": predicted_winner,

        "score_to_submit": score_to_submit["scoreline"],
        "score_to_submit_probability": round(score_to_submit["score_probability"], 4),

        "prob_home_win": round(home_win_prob, 4),
        "prob_draw": round(draw_prob, 4),
        "prob_away_win": round(away_win_prob, 4)
    })

future_score_predictions = pd.DataFrame(all_predictions)

# =========================
# Save
# =========================

os.makedirs("./results", exist_ok=True)

future_score_predictions.to_csv(
    "./results/world_cup_score_predictions_poisson_optimised.csv",
    index=False
)

print(future_score_predictions.head(20))

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best home-goal alpha:
{'model__alpha': 0.05}
Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best away-goal alpha:
{'model__alpha': 0.01}
Home goals MAE: 1.0626883380352883
Away goals MAE: 0.8343422131526068
         date      home_team               away_team      tournament  neutral  \
0  2026-06-11    South Korea          Czech Republic  FIFA World Cup        1   
1  2026-06-11         Mexico            South Africa  FIFA World Cup        0   
2  2026-06-12         Canada  Bosnia and Herzegovina  FIFA World Cup        0   
3  2026-06-12  United States                Paraguay  FIFA World Cup        0   
4  2026-06-13          Qatar             Switzerland  FIFA World Cup        1   
5  2026-06-13         Brazil                 Morocco  FIFA World Cup        1   
6  2026-06-13          Haiti                Scotland  FIFA World Cup        1   
7  2026-06-13      Australia                  Turkey  FIFA World Cup    

Ensemble model combining Poisson model to predict score probabilities, XGBoost predict outcome probabilities, then average the two probability sets.

Draw correction: world cup 2022: 27% draws in group stages, 2018: 31%

In [25]:
# =========================
# Preprocessor
# =========================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# =========================
# XGBoost model optimisation
# =========================

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_param_grid = {
    "model__n_estimators": [300],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.03, 0.05],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8],
    "model__min_child_weight": [1, 5]
}

tscv = TimeSeriesSplit(n_splits=3)

xgb_grid = GridSearchCV(
    xgb_pipeline,
    xgb_param_grid,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

xgb_grid.fit(X_train, y_train_enc)

xgb_model = xgb_grid.best_estimator_

print("\nBest XGBoost parameters:")
print(xgb_grid.best_params_)

xgb_test_probs = xgb_model.predict_proba(X_test)
xgb_test_pred = le.inverse_transform(
    np.argmax(xgb_test_probs, axis=1)
)

print("\nXGBoost test accuracy:")
print(accuracy_score(y_test, xgb_test_pred))

print("\nXGBoost classification report:")
print(classification_report(y_test, xgb_test_pred))

# =========================
# Poisson goal model optimisation
# =========================

poisson_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", PoissonRegressor(max_iter=2000))
])

poisson_param_grid = {
    "model__alpha": [0.001, 0.01, 0.05, 0.1, 0.5, 1, 2]
}

home_grid = GridSearchCV(
    poisson_pipeline,
    poisson_param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

home_grid.fit(X_train, train["home_score"])

home_goal_model = home_grid.best_estimator_

print("\nBest home-goal Poisson parameters:")
print(home_grid.best_params_)

away_grid = GridSearchCV(
    poisson_pipeline,
    poisson_param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

away_grid.fit(X_train, train["away_score"])

away_goal_model = away_grid.best_estimator_

print("\nBest away-goal Poisson parameters:")
print(away_grid.best_params_)

home_lambda_test = home_goal_model.predict(X_test)
away_lambda_test = away_goal_model.predict(X_test)

print("\nPoisson home goals MAE:")
print(mean_absolute_error(test["home_score"], home_lambda_test))

print("\nPoisson away goals MAE:")
print(mean_absolute_error(test["away_score"], away_lambda_test))

# =========================
# Helper functions
# =========================

def score_matrix(home_lambda, away_lambda, max_goals=8):
    rows = []

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):

            prob = (
                poisson.pmf(home_goals, home_lambda) *
                poisson.pmf(away_goals, away_lambda)
            )

            rows.append({
                "home_score_pred": home_goals,
                "away_score_pred": away_goals,
                "scoreline": f"{home_goals}-{away_goals}",
                "score_probability": prob
            })

    scores = pd.DataFrame(rows)

    scores["score_probability"] = (
        scores["score_probability"] /
        scores["score_probability"].sum()
    )

    return scores.sort_values(
        "score_probability",
        ascending=False
    )


def poisson_outcome_probs(home_lambda, away_lambda, max_goals=8):

    scores = score_matrix(
        home_lambda,
        away_lambda,
        max_goals=max_goals
    )

    prob_home_win = scores[
        scores["home_score_pred"] > scores["away_score_pred"]
    ]["score_probability"].sum()

    prob_draw = scores[
        scores["home_score_pred"] == scores["away_score_pred"]
    ]["score_probability"].sum()

    prob_away_win = scores[
        scores["home_score_pred"] < scores["away_score_pred"]
    ]["score_probability"].sum()

    return prob_home_win, prob_draw, prob_away_win, scores

# =========================
# Tune ensemble weights
# =========================

weight_results = []

def blend_probabilities(xgb_probs, poisson_probs, weight_xgb, draw_boost=1):
    weight_poisson = 1 - weight_xgb

    blended = {
        cls: (
            weight_xgb * xgb_probs[cls] +
            weight_poisson * poisson_probs[cls]
        )
        for cls in xgb_probs
    }

    blended["Draw"] *= draw_boost

    total = sum(blended.values())

    return {
        cls: prob / total
        for cls, prob in blended.items()
    }

for weight_xgb in np.arange(0, 1.01, 0.05):

    weight_poisson = 1 - weight_xgb
    ensemble_test_predictions = []

    for i in range(len(X_test)):

        p_home, p_draw, p_away, _ = poisson_outcome_probs(
            home_lambda_test[i],
            away_lambda_test[i],
            max_goals=8
        )

        poisson_probs = {
            "AwayWin": p_away,
            "Draw": p_draw,
            "HomeWin": p_home
        }

        xgb_probs = {
            le.classes_[j]: xgb_test_probs[i, j]
            for j in range(len(le.classes_))
        }

        blended_probs = blend_probabilities(
            xgb_probs,
            poisson_probs,
            weight_xgb,
            draw_boost=1
        )

        ensemble_test_predictions.append(
            max(blended_probs, key=blended_probs.get)
        )

    acc = accuracy_score(y_test, ensemble_test_predictions)

    weight_results.append({
        "weight_xgb": weight_xgb,
        "weight_poisson": weight_poisson,
        "accuracy": acc
    })

weight_results = pd.DataFrame(weight_results)

print("\nBest ensemble weights:")
print(weight_results.sort_values("accuracy", ascending=False).head(10))

best_weight_row = weight_results.sort_values(
    "accuracy",
    ascending=False
).iloc[0]

WEIGHT_XGB = best_weight_row["weight_xgb"]
WEIGHT_POISSON = best_weight_row["weight_poisson"]

print("\nChosen XGBoost weight:", WEIGHT_XGB)
print("Chosen Poisson weight:", WEIGHT_POISSON)

# =========================
# Final ensemble test performance
# =========================

final_ensemble_test_predictions = []

for i in range(len(X_test)):

    p_home, p_draw, p_away, _ = poisson_outcome_probs(
        home_lambda_test[i],
        away_lambda_test[i],
        max_goals=8
    )

    poisson_probs = {
        "AwayWin": p_away,
        "Draw": p_draw,
        "HomeWin": p_home
    }

    xgb_probs = {
        le.classes_[j]: xgb_test_probs[i, j]
        for j in range(len(le.classes_))
    }

    blended_probs = {
        cls: (
            WEIGHT_XGB * xgb_probs[cls] +
            WEIGHT_POISSON * poisson_probs[cls]
        )
        for cls in le.classes_
    }

    final_ensemble_test_predictions.append(
        max(blended_probs, key=blended_probs.get)
    )

print("\nFinal ensemble test accuracy:")
print(accuracy_score(y_test, final_ensemble_test_predictions))

print("\nFinal ensemble classification report:")
print(classification_report(y_test, final_ensemble_test_predictions))

# =========================
# Predict future matches
# =========================

xgb_future_probs = xgb_model.predict_proba(X_future)

future_home_lambda = home_goal_model.predict(X_future)
future_away_lambda = away_goal_model.predict(X_future)

all_predictions = []

# Give draws a small boost, without, the model does not predict any draws
DRAW_BOOST = 1.25

for i, row in future_matches.reset_index(drop=True).iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]

    home_lambda = future_home_lambda[i]
    away_lambda = future_away_lambda[i]

    p_home_poisson, p_draw_poisson, p_away_poisson, scores = poisson_outcome_probs(
        home_lambda,
        away_lambda,
        max_goals=8
    )

    poisson_probs = {
        "AwayWin": p_away_poisson,
        "Draw": p_draw_poisson,
        "HomeWin": p_home_poisson
    }

    xgb_probs = {
        le.classes_[j]: xgb_future_probs[i, j]
        for j in range(len(le.classes_))
    }

    blended_probs = blend_probabilities(
        xgb_probs,
        poisson_probs,
        WEIGHT_XGB,
        draw_boost=DRAW_BOOST
    )

    blended_probs["Draw"] = blended_probs["Draw"] * DRAW_BOOST

    # Re-normalise probabilities so they still sum to 1
    total_prob = sum(blended_probs.values())

    blended_probs = {
        cls: prob / total_prob
        for cls, prob in blended_probs.items()
    }

    predicted_result = max(blended_probs, key=blended_probs.get)

    most_likely = scores.iloc[0]

    if predicted_result == "HomeWin":
        score_to_submit = scores[
            scores["home_score_pred"] >
            scores["away_score_pred"]
        ].iloc[0]

    elif predicted_result == "AwayWin":
        score_to_submit = scores[
            scores["home_score_pred"] <
            scores["away_score_pred"]
        ].iloc[0]

    else:
        score_to_submit = scores[
            scores["home_score_pred"] ==
            scores["away_score_pred"]
        ].iloc[0]

    predicted_winner = (
        home_team if predicted_result == "HomeWin"
        else away_team if predicted_result == "AwayWin"
        else "Draw"
    )

    all_predictions.append({
        "date": row["date"],
        "home_team": home_team,
        "away_team": away_team,
        "tournament": row["tournament"],
        "neutral": row["neutral"],

        "expected_home_goals": round(home_lambda, 2),
        "expected_away_goals": round(away_lambda, 2),
        "expected_score": f"{round(home_lambda, 1)}-{round(away_lambda, 1)}",

        "most_likely_score": most_likely["scoreline"],
        "most_likely_score_probability": round(most_likely["score_probability"], 4),

        "predicted_result": predicted_result,
        "predicted_winner": predicted_winner,

        "score_to_submit": score_to_submit["scoreline"],
        "score_to_submit_probability": round(score_to_submit["score_probability"], 4),

        "xgb_prob_home_win": round(xgb_probs["HomeWin"], 4),
        "xgb_prob_draw": round(xgb_probs["Draw"], 4),
        "xgb_prob_away_win": round(xgb_probs["AwayWin"], 4),

        "poisson_prob_home_win": round(poisson_probs["HomeWin"], 4),
        "poisson_prob_draw": round(poisson_probs["Draw"], 4),
        "poisson_prob_away_win": round(poisson_probs["AwayWin"], 4),

        "ensemble_prob_home_win": round(blended_probs["HomeWin"], 4),
        "ensemble_prob_draw": round(blended_probs["Draw"], 4),
        "ensemble_prob_away_win": round(blended_probs["AwayWin"], 4),

        "ensemble_confidence": round(max(blended_probs.values()), 4),

        "weight_xgb": round(WEIGHT_XGB, 2),
        "weight_poisson": round(WEIGHT_POISSON, 2)
    })

ensemble_predictions = pd.DataFrame(all_predictions)

# =========================
# Save
# =========================

os.makedirs("./results", exist_ok=True)

ensemble_predictions.to_csv(
    "./results/world_cup_predictions_optimised_ensemble_xgb_poisson.csv",
    index=False
)

print("\nSaved:")
print("./results/world_cup_predictions_optimised_ensemble_xgb_poisson.csv")

print("\nFuture predictions:")
print(ensemble_predictions["predicted_result"].value_counts())
print(ensemble_predictions.head(20))

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=5, model__n_estimators=300, model__subsample=0.8; total time=   1.0s
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=300, model__subsample=0.8; total time=   1.2s
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=300, model__subsample=0.8; total time=   1.8s
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=5, model__n_estimators=300, model__subsample=0.8; total time=   1.8s
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=5, model__min_child_weight=1, model__n_estimators=300, model__subsample=0.8; total time=   1.9s
[CV] END model__colsample_bytree=0.8, model__learn

/Users/lindseyvanhaute/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/lindseyvanhaute/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/lindseyvanhaute/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital


Saved:
./results/world_cup_predictions_optimised_ensemble_xgb_poisson.csv

Future predictions:
predicted_result
HomeWin    33
Draw       20
AwayWin    19
Name: count, dtype: int64
         date      home_team               away_team      tournament  neutral  \
0  2026-06-11    South Korea          Czech Republic  FIFA World Cup        1   
1  2026-06-11         Mexico            South Africa  FIFA World Cup        0   
2  2026-06-12         Canada  Bosnia and Herzegovina  FIFA World Cup        0   
3  2026-06-12  United States                Paraguay  FIFA World Cup        0   
4  2026-06-13          Qatar             Switzerland  FIFA World Cup        1   
5  2026-06-13         Brazil                 Morocco  FIFA World Cup        1   
6  2026-06-13          Haiti                Scotland  FIFA World Cup        1   
7  2026-06-13      Australia                  Turkey  FIFA World Cup        1   
8  2026-06-14        Germany                 Curaçao  FIFA World Cup        1   
9  2026-0